In [1]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
)
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/propertyhub')
START_URL = "https://propertyhub.in.th/%E0%B8%82%E0%B8%B2%E0%B8%A2%E0%B8%84%E0%B8%AD%E0%B8%99%E0%B9%82%E0%B8%94"
OUTPUT_CSV_FILE = BASE_DIR / 'propertyhub_listing_urls.csv'

MAX_PAGES = 20
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return uc.Chrome(options=options, version_main=145)


def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('li[data-za-section="listing_card"] a[href]'))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]


def scroll_to_bottom(driver: uc.Chrome):
    last_h = 0
    for _ in range(6):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.4)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h


def click_next_page(driver: uc.Chrome, wait: WebDriverWait) -> bool:
    try:
        xpath = "//a[contains(text(), 'ถัดไป') or contains(text(), 'Next')]"
        next_btn = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))

        if next_btn.get_attribute("aria-disabled") == "true":
            return False

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", next_btn)
        return True
    except (TimeoutException, NoSuchElementException, ElementClickInterceptedException):
        return False


def get_first_listing_href(driver: uc.Chrome) -> str:
    js = "return document.querySelector('li[data-za-section=\"listing_card\"] a[href]')?.getAttribute('href') || '';"
    return driver.execute_script(js)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    parsed_url = urlparse(START_URL)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"

    all_urls = set()
    current_page = 1

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(START_URL)

        while current_page <= MAX_PAGES:
            logging.info(f"Processing Page {current_page}/{MAX_PAGES}...")

            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'li[data-za-section="listing_card"]')))
            except TimeoutException:
                logging.warning(f"Timeout waiting for elements on page {current_page}. Stopping.")
                break

            scroll_to_bottom(driver)

            new_links = extract_links(driver, base_url)
            if not new_links:
                logging.info("No links found on this page. Stopping.")
                break

            all_urls.update(new_links)
            logging.info(f"Found {len(new_links)} listings (Total unique: {len(all_urls)})")

            if current_page >= MAX_PAGES:
                logging.info(f"Reached target max pages ({MAX_PAGES}).")
                break

            current_first_href = get_first_listing_href(driver)
            current_url = driver.current_url

            if not click_next_page(driver, wait):
                logging.info("No 'Next' button found or it is disabled. Reached the last page.")
                break

            try:
                wait.until(
                    lambda d: d.current_url != current_url or get_first_listing_href(d) != current_first_href
                )
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'li[data-za-section="listing_card"]')))
            except TimeoutException:
                logging.error("Timeout waiting for the next page to load. Stopping.")
                break

            current_page += 1

    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in sorted(all_urls):
            writer.writerow([u])

    logging.info(f"Successfully saved {len(all_urls)} URLs to {OUTPUT_CSV_FILE}")


if __name__ == "__main__":
    main()

2026-03-29 15:57:12,352 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 15:57:13,392 - INFO - Starting scrape at: https://propertyhub.in.th/%E0%B8%82%E0%B8%B2%E0%B8%A2%E0%B8%84%E0%B8%AD%E0%B8%99%E0%B9%82%E0%B8%94
2026-03-29 15:57:27,657 - INFO - Processing Page 1/20...
2026-03-29 15:57:28,537 - INFO - Found 119 listings (Total unique: 119)
2026-03-29 15:57:44,415 - ERROR - Timeout waiting for the next page to load. Stopping.
2026-03-29 15:57:44,535 - INFO - Successfully saved 119 URLs to /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/propertyhub/propertyhub_listing_urls.csv


In [2]:
import csv
import logging
import time
from pathlib import Path

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/propertyhub')
INPUT_CSV = BASE_DIR / 'propertyhub_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'propertyhub_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    
    return uc.Chrome(options=options, version_main=145)


def extract_content(driver: uc.Chrome) -> str:
    data = []
    
    try:
        h1 = driver.find_element(By.TAG_NAME, 'h1')
        data.extend(["--- Listing Title ---", h1.text.strip()])
    except NoSuchElementException:
        pass

    try:
        price = driver.find_element(By.CSS_SELECTOR, '.sale-price')
        data.extend(["\n--- Price ---", price.text.strip()])
    except NoSuchElementException:
        pass

    try:
        specs_container = driver.find_element(By.XPATH, "//div[.//svg and .//span[contains(text(), 'ตร.ม.') or contains(text(), 'ตร.ว.') or contains(text(), 'นอน')]]")
        specs = specs_container.find_elements(By.XPATH, "./div")
        if specs:
            data.append("\n--- Basic Specs ---")
            data.append(" | ".join([s.text.strip() for s in specs if s.text.strip()]))
    except NoSuchElementException:
        pass

    try:
        details_list = driver.find_elements(By.XPATH, "//h3[contains(text(), 'รายละเอียด')]/following-sibling::ul/li")
        if details_list:
            data.append("\n--- Details ---")
            for li in details_list:
                data.append(li.text.replace('\n', ': ').strip())
    except NoSuchElementException:
        pass

    try:
        expand_btn = driver.find_element(By.XPATH, "//span[contains(text(), 'ดูข้อมูลประกาศทั้งหมด')]")
        driver.execute_script("arguments[0].click();", expand_btn)
        time.sleep(0.3)
    except NoSuchElementException:
        pass

    try:
        desc_elements = driver.find_elements(By.XPATH, "//div[span[contains(text(), 'ย่อข้อมูลประกาศ')]]/preceding-sibling::div")
        if desc_elements:
            data.extend(["\n--- Description ---", desc_elements[0].text.strip()])
    except NoSuchElementException:
        pass

    return "\n".join(data)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 10)

    try:
        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    if driver.current_url != url:
                        driver.get(url)
                    
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

2026-03-29 15:59:26,904 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:00:00,639 - INFO - Progress: [10/119] scraped.
2026-03-29 16:00:39,082 - INFO - Progress: [20/119] scraped.
2026-03-29 16:01:14,798 - INFO - Progress: [30/119] scraped.
2026-03-29 16:01:49,976 - INFO - Progress: [40/119] scraped.
2026-03-29 16:02:18,205 - INFO - Progress: [50/119] scraped.
2026-03-29 16:03:00,943 - INFO - Progress: [60/119] scraped.
2026-03-29 16:03:14,041 - INFO - Progress: [70/119] scraped.
2026-03-29 16:03:24,564 - INFO - Progress: [80/119] scraped.
2026-03-29 16:03:35,355 - INFO - Progress: [90/119] scraped.
2026-03-29 16:03:48,523 - INFO - Progress: [100/119] scraped.
2026-03-29 16:04:02,760 - INFO - Progress: [110/119] scraped.
2026-03-29 16:04:16,794 - INFO - Scraping completed. Saved to: /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/propertyhub/propertyhub_full_details.csv
